In [1]:
import pandas as pd
import numpy as np
from scipy.stats import dirichlet

In [2]:
path = "./Datasets/final/scen-training.csv"

### Read sampled dataset

In [3]:
df = pd.read_csv(f"{path}")

In [4]:
print(df.shape)

(2344433, 41)


### Define number of clients and alpha value

In [5]:
clients = 10
alpha = 0.5  # smaller alpha → more skewed, larger alpha → closer to IID

### Separate indices by label

In [6]:
label_indices = {
    label: df[df['label_encoded'] == label].index.tolist()
    for label in df['label_encoded'].unique()
}

In [7]:
client_indices = [[] for _ in range(clients)]

### Assign samples to clients using Dirichlet distribution

In [8]:
for label, indices in label_indices.items():
    np.random.shuffle(indices)

    # Generate Dirichlet proportions using SciPy
    proportions = dirichlet.rvs([alpha] * clients, size=1)[0]

    # Convert proportions to actual counts
    proportions = (proportions * len(indices)).astype(int)

    # Fix rounding errors
    diff = len(indices) - proportions.sum()
    for i in range(diff):
        proportions[i % clients] += 1

    # Assign samples to clients
    start = 0
    for i, count in enumerate(proportions):
        client_indices[i].extend(indices[start:start + count])
        start += count

#### Save datasets

In [9]:
for i, idx in enumerate(client_indices):
    client_df = df.loc[idx].reset_index(drop=True)
    client_df.to_csv(f"./Datasets/nonIID/client_{i+1}_nonIID.csv", index=False)
    print(f"Client {i+1} -> {client_df['label_encoded'].value_counts().to_dict()}")

Client 1 -> {1: 33835, 0: 4135}
Client 2 -> {1: 37485, 0: 12043}
Client 3 -> {1: 21153, 0: 3180}
Client 4 -> {1: 975419, 0: 6934}
Client 5 -> {0: 8232, 1: 2964}
Client 6 -> {1: 186738, 0: 2994}
Client 7 -> {1: 525907, 0: 15973}
Client 8 -> {1: 242787, 0: 16}
Client 9 -> {1: 33619, 0: 56}
Client 10 -> {1: 216328, 0: 14635}
